In [11]:
#import glob
#import os

# Path to the folder containing the Excel files
#folder_path = "cleaned"

# Load all Excel files in the folder
#excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

# Initialize an empty list to store the DataFrames
#df_list = []

# Loop through each file and read the Excel sheet into a DataFrame
#for file in excel_files:
    #df = pd.read_excel(file)
    #df_list.append(df)

# Combine all DataFrames by appending them
#combined_df = pd.concat(df_list, ignore_index=True)

# Display the combined DataFrame
#print(combined_df)

# Optionally, save the combined data to a new Excel file
#combined_df.to_csv("combined_property_data.csv", index=False)

In [ ]:
!pip uninstall numpy

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

In [ ]:
# Download NLTK data files (run this once if not downloaded)
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
# Load data
property_df = pd.read_csv('20240911v properties with counts.csv')  # property DataFrame
zv_df = pd.read_csv('combined_property_data.csv') # Zonal Values DataFrame

# Set up stop words and punctuation for preprocessing
stop_words = set(stopwords.words('english'))
punctuation = set(string.punctuation)

In [ ]:
# Define the classification function for Property Type in property_df
def classify_property_type(category):
    category = category.lower()
    if 'house' in category or 'condo' in category:
        return 'RR'
    elif 'commercial' in category or 'land' in category:
        return 'CC'
    return 'Unknown'  # In case there are other categories

# Apply the classification function to create Property Type column in property_df
property_df['Property Type'] = property_df['Category'].apply(classify_property_type)

In [ ]:
# Function to preprocess location text (remove unwanted characters, tokenize)
def preprocess_location(location):
    # Tokenize and clean text: remove special characters and stop words
    tokens = word_tokenize(location.lower())
    cleaned_tokens = [token for token in tokens if token not in stop_words and token not in punctuation]
    cleaned_text = ' '.join(cleaned_tokens)
    return cleaned_text

# Function to preprocess location text (remove unwanted characters, tokenize)
def preprocess_location(location):
    tokens = word_tokenize(location.lower())
    cleaned_tokens = [token for token in tokens if token not in stop_words and token not in punctuation]
    cleaned_text = ' '.join(cleaned_tokens)
    return cleaned_text

In [ ]:
# Preprocess the locations
property_df['Processed_Location'] = property_df['Location'].apply(preprocess_location)
zv_df['Processed_Location'] = (zv_df['Street'] + " " + zv_df['Vicinity'] + " " + 
                               zv_df['Barangay'] + " " + zv_df['City'] + " " + zv_df['Province']).apply(preprocess_location)

# Create a TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Combine both property and zonal value locations
combined_locations = pd.concat([property_df['Processed_Location'], zv_df['Processed_Location']], ignore_index=True)

In [ ]:
# Fit and transform the combined locations
tfidf_matrix = vectorizer.fit_transform(combined_locations)

# Split the TF-IDF matrix for property and zonal value locations
property_tfidf = tfidf_matrix[:len(property_df)]
zv_tfidf = tfidf_matrix[len(property_df):]

In [3]:
# Calculate cosine similarity between property locations and zonal value locations
cosine_similarities = cosine_similarity(property_tfidf, zv_tfidf)

# Assign the most similar zonal value based on both classification and similarity match
def get_best_zonal_match(similarities, zv_df, prop_classification):
    # Filter zonal values based on matching Property Type and Classification
    eligible_zonal_values = property_df[property_df['Property Type'] == prop_classification]
    if eligible_zonal_values.empty:
        return None  # No match found if no matching classifications
    
    # Use similarity scores only for the eligible zonal values
    eligible_similarities = similarities[eligible_zonal_values.index]
    best_match_idx = eligible_similarities.argmax()
    return eligible_zonal_values.iloc[best_match_idx]['Zonal Value']


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Dianne Yumol\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\Dianne Y

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
# Apply the matching process
property_df['Matched_Zonal_Value'] = [
    get_best_zonal_match(cosine_similarities[i], zv_df, row['Classification'])
    for i, row in property_df.iterrows()
]

In [ ]:
# Show the results
print(property_df[['Location', 'Classification', 'Matched_Zonal_Value']])